In [20]:
import pandas as pd
import numpy as np
import sklearn as sk

1. Загрузите данные из файла data-logistic.csv. Это двумерная выборка, целевая переменная на которой принимает значения -1 или 1.

In [21]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ML_intro/08/data-logistic.csv', header=0, names=['target', 'f1', 'f2'])
y = data['target'].to_numpy()
X = data.drop(columns=['target']).to_numpy()

2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска. Обратите внимание, что мы используем полноценный градиентный спуск, а не его стохастический вариант!


3. Реализуйте градиентный спуск для обычной и L2-регуляризованной
(с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения
используйте вектор (0, 0).


In [22]:
k = 0.0
class GradDecs():
  def __init__(self, X, y) -> None:
    self.k = k
    self.X = X
    self.y = y
    self.l = len(y)

  def grad_step(self, w, k, C):
    w1 = w[0]
    w2 = w[1]

    w1 = w1 + k*(1/self.l) * sum([
        self.y[i]*self.X[i][0] * (
            1 - 1/(1 + np.exp(-self.y[i] * (w1*self.X[i][0] + w2*self.X[i][1])))
        ) - k*C*w1
    for i in range(len(y))])

    w2 = w2 + k*(1/self.l) * sum([
        self.y[i]*self.X[i][1] * (
            1 - 1/(1 + np.exp(-self.y[i] * (w1*self.X[i][0] + w2*self.X[i][1])))
        ) - k*C*w2
    for i in range(len(y))])

    w = np.array([w1, w2])
    return w

  def log_reg(self, C, k, max_steps, w_start):
    conv = False
    iter_num = 0
    w = w_start
    print("Начальное приближение w: ", w_start)
    for _ in range(max_steps):
      w1, w2 = w[0], w[1]

      # s = (1/self.l) * sum([
      #     np.log(
      #         1 + np.exp(-yi * (w1*xi1 + w2*xi2))
      #     ) + 1/2 * C * (np.linalg.norm(w))**2
      # ])

      nextw = self.grad_step(w, k, C)
      if np.linalg.norm(nextw-w) <= 10**(-5):
        conv = True
        break
      iter_num += 1
      w = nextw
    print("Шаг: ", round(k, 1))
    if conv == False:
      print("Алгоритм не сошелся с нужной точностью")
    if C == 0:
      print("Количество итераций без регуляризации: ", iter_num)
    else:
      print("Количество итераций с регуляризацией: ", iter_num)
    return w

4. Запустите градиентный спуск и доведите до сходимости (евклидово
расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). Рекомендуется ограничить сверху число
итераций десятью тысячами.


5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? Эти величины будут ответом на
задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом.
Для этого воспользуйтесь сигмоидной функцией: a(x) = 1/(1 +
exp(−w1x1 − w2x2)).

In [23]:
GD = GradDecs(X, y)

w = GD.log_reg(0, 0.1, 10000, np.array([0, 0]))
a = [1/(1 + np.exp(-w[0] * xi[0] - w[1] * xi[1])) for xi in X]
score = sk.metrics.roc_auc_score(y, a)
print("Значение AUC-ROC без регуляризации: ", round(score, 3), '\n')

wL2 = GD.log_reg(10, 0.1, 10000, np.array([0, 0]))
aL2 = [1/(1 + np.exp(-wL2[0] * xi[0] - wL2[1] * xi[1])) for xi in X]
scoreL2 = sk.metrics.roc_auc_score(y, aL2)
print("Значение AUC-ROC с регуляризацией: ", round(scoreL2, 3), '\n')

Начальное приближение w:  [0 0]
Шаг:  0.1
Количество итераций без регуляризации:  232
Значение AUC-ROC без регуляризации:  0.927 

Начальное приближение w:  [0 0]
Шаг:  0.1
Количество итераций с регуляризацией:  38
Значение AUC-ROC с регуляризацией:  0.937 



6. Попробуйте поменять длину шага. Будет ли сходиться алгоритм,
если делать более длинные шаги? Как меняется число итераций
при уменьшении длины шага?


In [24]:
for k in [0.1 * n for n in range(1, 6)]:
  try:
    w = GD.log_reg(0, k, 10000, np.array([0, 0]))
    a = [1/(1 + np.exp(-w[0] * xi[0] - w[1] * xi[1])) for xi in X]
    score = sk.metrics.roc_auc_score(y, a)
    print("Значение AUC-ROC без регуляризации: ", round(score, 3), '\n')

    wL2 = GD.log_reg(10, k, 10000, np.array([0, 0]))
    aL2 = [1/(1 + np.exp(-wL2[0] * xi[0] - wL2[1] * xi[1])) for xi in X]
    scoreL2 = sk.metrics.roc_auc_score(y, aL2)
    print("Значение AUC-ROC с регуляризацией: ", round(scoreL2, 3), '\n')
  except Exception as e:
    print(e)
    print("Алгоритм не сошелся с нужной точностью")

Начальное приближение w:  [0 0]
Шаг:  0.1
Количество итераций без регуляризации:  232
Значение AUC-ROC без регуляризации:  0.927 

Начальное приближение w:  [0 0]
Шаг:  0.1
Количество итераций с регуляризацией:  38
Значение AUC-ROC с регуляризацией:  0.937 

Начальное приближение w:  [0 0]
Шаг:  0.2
Количество итераций без регуляризации:  124
Значение AUC-ROC без регуляризации:  0.927 

Начальное приближение w:  [0 0]
Шаг:  0.2
Количество итераций с регуляризацией:  10
Значение AUC-ROC с регуляризацией:  0.937 

Начальное приближение w:  [0 0]
Шаг:  0.3
Количество итераций без регуляризации:  83
Значение AUC-ROC без регуляризации:  0.927 

Начальное приближение w:  [0 0]
Шаг:  0.3
Количество итераций с регуляризацией:  8
Значение AUC-ROC с регуляризацией:  0.937 

Начальное приближение w:  [0 0]
Шаг:  0.4
Количество итераций без регуляризации:  61
Значение AUC-ROC без регуляризации:  0.927 

Начальное приближение w:  [0 0]
Шаг:  0.4
Алгоритм не сошелся с нужной точностью
Количество ите

/tmp/ipykernel_19408/4186940689.py:21: RuntimeWarning: overflow encountered in exp
  1 - 1/(1 + np.exp(-self.y[i] * (w1*self.X[i][0] + w2*self.X[i][1])))
/tmp/ipykernel_19408/4186940689.py:15: RuntimeWarning: overflow encountered in exp
  1 - 1/(1 + np.exp(-self.y[i] * (w1*self.X[i][0] + w2*self.X[i][1])))
/tmp/ipykernel_19408/4186940689.py:19: RuntimeWarning: overflow encountered in scalar add
  w2 = w2 + k*(1/self.l) * sum([
/tmp/ipykernel_19408/4186940689.py:13: RuntimeWarning: overflow encountered in scalar add
  w1 = w1 + k*(1/self.l) * sum([
/tmp/ipykernel_19408/4186940689.py:21: RuntimeWarning: invalid value encountered in scalar add
  1 - 1/(1 + np.exp(-self.y[i] * (w1*self.X[i][0] + w2*self.X[i][1])))


Шаг:  0.5
Алгоритм не сошелся с нужной точностью
Количество итераций с регуляризацией:  10000
Input contains NaN.
Алгоритм не сошелся с нужной точностью


*Вывод:* при увеличении шага количество итераций уменьшается, но на каком-то моменте алгоритм перестает сходиться. То есть нужно подбирать параметр k так, чтобы минимизировать количество итераций при условии сходимости алгоритма.

7. Попробуйте менять начальное приближение. Влияет ли оно на чтонибудь?


In [25]:
ws = []
for n in range(1, 11):
  ws.append([1*n, 1*n])
ws = np.array(ws)
for w_start in ws:
  w = GD.log_reg(0, 0.1, 10000, w_start)
  a = [1/(1 + np.exp(-w[0] * xi[0] - w[1] * xi[1])) for xi in X]
  score = sk.metrics.roc_auc_score(y, a)
  print("Значение AUC-ROC без регуляризации: ", round(score, 3), '\n')

  wL2 = GD.log_reg(10, 0.1, 10000, w_start)
  aL2 = [1/(1 + np.exp(-wL2[0] * xi[0] - wL2[1] * xi[1])) for xi in X]
  scoreL2 = sk.metrics.roc_auc_score(y, aL2)
  print("Значение AUC-ROC с регуляризацией: ", round(scoreL2, 3), '\n')

Начальное приближение w:  [1 1]
Шаг:  0.1
Количество итераций без регуляризации:  229
Значение AUC-ROC без регуляризации:  0.927 

Начальное приближение w:  [1 1]
Шаг:  0.1
Количество итераций с регуляризацией:  48
Значение AUC-ROC с регуляризацией:  0.937 

Начальное приближение w:  [2 2]
Шаг:  0.1
Количество итераций без регуляризации:  154
Значение AUC-ROC без регуляризации:  0.927 

Начальное приближение w:  [2 2]
Шаг:  0.1
Количество итераций с регуляризацией:  51
Значение AUC-ROC с регуляризацией:  0.937 

Начальное приближение w:  [3 3]
Шаг:  0.1
Количество итераций без регуляризации:  278
Значение AUC-ROC без регуляризации:  0.927 

Начальное приближение w:  [3 3]
Шаг:  0.1
Количество итераций с регуляризацией:  53
Значение AUC-ROC с регуляризацией:  0.937 

Начальное приближение w:  [4 4]
Шаг:  0.1
Количество итераций без регуляризации:  326
Значение AUC-ROC без регуляризации:  0.927 

Начальное приближение w:  [4 4]
Шаг:  0.1
Количество итераций с регуляризацией:  54
Значение

*Вывод:* чем дальше начальное приближение от ответа, тем больше итераций уйдет на сходимость алгоритма.